# 1.0 Sobre o Notebook

- O objetivo deste notebook é fazer a limpeza e organização dos dados referentes ao dataset Olist
- Fonte dos dados: https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce
- As tabelas com dados brutos encontram-se na pasta data/raw/
- As tabelas limpas estão em data/processed/ 

# 2.0 Problema de Negócio

- A Olist apresenta crescimento nas vendas e satisfação de clientes que justifiquem investimentos na expansão da plataforma?
    - As vendas estão crescendo? -> Analisar a evolução mensal dos pedidos e valor vendido
    - Os clientes estão satisfeitos? -> Analisar review score
    - O que prejudica a satisfação de um cliente? -> Analisar prazo de entrega x review_score
    - Onde estão as oportunidades? -> Olhar categorias e regiões com melhor/pior desempenho

# 3.0 Imports

In [1]:
import pandas as pd

# 4.0 Carregamento dos Dados Brutos

In [2]:
df_pedidos = pd.read_csv('../data/raw/olist_orders_dataset.csv')
df_clientes = pd.read_csv('../data/raw/olist_customers_dataset.csv')
df_itens = pd.read_csv('../data/raw/olist_order_items_dataset.csv')
df_avaliacoes = pd.read_csv('../data/raw/olist_order_reviews_dataset.csv')
df_produtos = pd.read_csv('../data/raw/olist_products_dataset.csv')

# 5.0 Limpeza das Tabelas

## 5.1 Tabela de Pedidos

In [3]:
# criando cópia dos dados raw para aplicar a limpeza
df_pedidos_clean = df_pedidos.copy()

In [4]:
# apagando colunas que não serão utilizadas
df_pedidos_clean = df_pedidos_clean.drop(columns=['order_approved_at', 'order_delivered_carrier_date'])

In [5]:
# checando limpeza
print(f'Dados Brutos: {df_pedidos.shape}')
print(f'Dados Limpos: {df_pedidos_clean.shape}')

Dados Brutos: (99441, 8)
Dados Limpos: (99441, 6)


In [6]:
# convertendo as colunas de data
df_pedidos_clean['order_purchase_timestamp'] = pd.to_datetime(df_pedidos_clean['order_purchase_timestamp'])
df_pedidos_clean['order_delivered_customer_date'] = pd.to_datetime(df_pedidos_clean['order_delivered_customer_date'])
df_pedidos_clean['order_estimated_delivery_date'] = pd.to_datetime(df_pedidos_clean['order_estimated_delivery_date'])

In [7]:
# checando ajuste das datas
print(f'Dados Brutos:\n {df_pedidos.dtypes}')
print()
print(f'Dados Limpos:\n {df_pedidos_clean.dtypes}')

Dados Brutos:
 order_id                         object
customer_id                      object
order_status                     object
order_purchase_timestamp         object
order_approved_at                object
order_delivered_carrier_date     object
order_delivered_customer_date    object
order_estimated_delivery_date    object
dtype: object

Dados Limpos:
 order_id                                 object
customer_id                              object
order_status                             object
order_purchase_timestamp         datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object


In [8]:
df_pedidos_clean['order_status'].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [9]:
# filtrando o dataset para manter apenas o status 'delivered'

df_pedidos_clean = df_pedidos_clean[df_pedidos_clean['order_status'] == 'delivered']

In [10]:
df_pedidos_clean['order_status'].value_counts()

order_status
delivered    96478
Name: count, dtype: int64

In [11]:
# checando limpeza
print(f'Dados Brutos: {df_pedidos.shape}')
print(f'Dados Limpos: {df_pedidos_clean.shape}')

Dados Brutos: (99441, 8)
Dados Limpos: (96478, 6)


In [12]:
# filtrando o período de análise de 2017-01 a 2018-08 (20 meses)
data_inicial = '2017-01-01 00:00:00'
data_final = '2018-08-31 23:59:59'

df_pedidos_clean = df_pedidos_clean[df_pedidos_clean['order_purchase_timestamp'].between(data_inicial, data_final)]

In [13]:
# analisando o período mês a mês após limpeza das pontas
temp_meses = df_pedidos_clean['order_purchase_timestamp']
print(f'Data inicial: {temp_meses.min()} e Data final: {temp_meses.max()}')

Data inicial: 2017-01-05 11:56:06 e Data final: 2018-08-29 15:00:37


In [14]:
# quantidade de pedidos mês a mês em ordem cronológica
temp_meses.dt.to_period('M').value_counts().sort_index()

order_purchase_timestamp
2017-01     750
2017-02    1653
2017-03    2546
2017-04    2303
2017-05    3546
2017-06    3135
2017-07    3872
2017-08    4193
2017-09    4150
2017-10    4478
2017-11    7289
2017-12    5513
2018-01    7069
2018-02    6555
2018-03    7003
2018-04    6798
2018-05    6749
2018-06    6099
2018-07    6159
2018-08    6351
Freq: M, Name: count, dtype: int64

In [15]:
# analisando os 8 nulos que haviam na coluna data de entrega para o cliente

data_nula = df_pedidos_clean['order_delivered_customer_date'].isna()
df_pedidos_clean.loc[data_nula, 'order_status'].value_counts()

order_status
delivered    8
Name: count, dtype: int64

In [16]:
# checando limpeza
print(f'Dados Brutos: {df_pedidos.shape}')
print(f'Dados Limpos: {df_pedidos_clean.shape}')

Dados Brutos: (99441, 8)
Dados Limpos: (96211, 6)


In [17]:
# salvando em data/processed
df_pedidos_clean.to_csv('../data/processed/pedidos_clean.csv', index=False)

### Registro da Limpeza - Tabela de Pedidos
- Colunas removidas: *order_approved_at*, *order_delivered_carrier_date* (8 -> 6 colunas)
- Datas convertidas: 3 colunas
- Filtro de status 'delivered': 99.441 -> 96.478 pedidos (-2.963)
- Filtro de período (jan/2017 a ago/2018): 96.478 -> 96.211 (-267)
    - Recorte confirmado após o filtro de status -> volume mensal consistente de jan/2017 a ago/2018 (6.351 pedidos em ago/2018)
- Mantidos os 8 pedidos sem data de entrega (serão excluídos apenas na análise de prazo)

## 5.2 Tabela de Clientes

In [18]:
# criando cópia dos dados raw para aplicar a limpeza
df_clientes_clean = df_clientes.copy()

In [19]:
# filtrando os dados de acordo com a tabela de pedidos
mascara_clientes_pedidos = df_clientes_clean['customer_id'].isin(df_pedidos_clean['customer_id'])
df_clientes_clean = df_clientes_clean[mascara_clientes_pedidos]

In [20]:
# checando filtro (deve ir para 96.211 linhas igual a tabela de pedidos)
print(f'Dados Brutos: {df_clientes.shape}')
print(f'Dados Limpos: {df_clientes_clean.shape}')

Dados Brutos: (99441, 5)
Dados Limpos: (96211, 5)


In [21]:
# apagando colunas que não serão utilizadas
df_clientes_clean = df_clientes_clean.drop(columns=['customer_zip_code_prefix', 'customer_city'])

In [22]:
# checando limpeza
print(f'Dados Brutos: {df_clientes.shape}')
print(f'Dados Limpos: {df_clientes_clean.shape}')

Dados Brutos: (99441, 5)
Dados Limpos: (96211, 3)


In [23]:
# contagem de customer_unique_id antes e depois das limpezas
print(f"Antes da limpeza: {df_clientes['customer_unique_id'].nunique()}")
print(f"Após a limpeza: {df_clientes_clean['customer_unique_id'].nunique()}")

Antes da limpeza: 96096
Após a limpeza: 93104


In [24]:
# salvando em data/processed
df_clientes_clean.to_csv('../data/processed/clientes_clean.csv', index=False)

### Registro da Limpeza - Tabela de Clientes
- Colunas removidas: *customer_zip_code_prefix*, *customer_city* (5 -> 3 colunas)
- Filtro dos dados de acordo com a Tabela de Pedidos: 99.441 -> 96.211 (-3.230)
    - Confirmada relação 1:1 entre as duas tabelas
- Clientes únicos: 96.096 -> 93.104 (-2.992)

## 5.3 Tabela de Itens dos Pedidos

In [25]:
# criando cópia dos dados raw para aplicar a limpeza
df_itens_clean = df_itens.copy()

In [26]:
# filtrando os dados de acordo com a tabela de pedidos
mascara_itens_pedidos = df_itens_clean['order_id'].isin(df_pedidos_clean['order_id'])
df_itens_clean = df_itens_clean[mascara_itens_pedidos]

In [27]:
# checando filtro
print(f'Dados Brutos: {df_itens.shape}')
print(f'Dados Limpos: {df_itens_clean.shape}')

Dados Brutos: (112650, 7)
Dados Limpos: (109880, 7)


In [28]:
# checando order_ids únicos (deve ir para 96.211)
df_itens_clean['order_id'].nunique()

96211

In [29]:
# apagando colunas que não serão utilizadas
df_itens_clean = df_itens_clean.drop(columns=['seller_id', 'shipping_limit_date', 'freight_value'])

In [30]:
# checando limpeza
print(f'Dados Brutos: {df_itens.shape}')
print(f'Dados Limpos: {df_itens_clean.shape}')

Dados Brutos: (112650, 7)
Dados Limpos: (109880, 4)


In [31]:
# salvando em data/processed
df_itens_clean.to_csv('../data/processed/itens_clean.csv', index=False)

### Registro da Limpeza - Tabela de Itens dos Pedidos
- Colunas removidas: *seller_id*, *shipping_limit_date* e *freight_value* (7 -> 4 colunas)
- Filtro de acordo com a Tabela de Pedidos: 112.650 -> 109.880 linhas de itens (-2.770)
- Previsão confirmada: 96.211 *order_id* únicos, todos os pedidos entregues possuem itens

## 5.4 Tabela de Avaliações

In [32]:
# criando cópia dos dados raw para aplicar a limpeza
df_avaliacoes_clean = df_avaliacoes.copy()

In [33]:
# filtrando os dados de acordo com a tabela de pedidos
mascara_avaliacoes_pedidos = df_avaliacoes_clean['order_id'].isin(df_pedidos_clean['order_id'])
df_avaliacoes_clean = df_avaliacoes_clean[mascara_avaliacoes_pedidos]

In [34]:
# checando filtro
print(f'Dados Brutos: {df_avaliacoes.shape}')
print(f'Dados Limpos: {df_avaliacoes_clean.shape}')

Dados Brutos: (99224, 7)
Dados Limpos: (96095, 7)


In [35]:
# checando order_id únicos
df_avaliacoes_clean['order_id'].nunique()

95568

In [36]:
# pedidos que contém mais de uma avaliação
contagem_avaliacoes = df_avaliacoes_clean['order_id'].value_counts()
mascara_mais_avaliacoes = contagem_avaliacoes > 1
pedidos_mais_avaliacoes = contagem_avaliacoes[mascara_mais_avaliacoes].index
len(pedidos_mais_avaliacoes)

523

In [37]:
# removendo avaliações excedentes
mascara_uma_avaliacao = ~df_avaliacoes_clean['order_id'].isin(pedidos_mais_avaliacoes)
df_avaliacoes_clean = df_avaliacoes_clean[mascara_uma_avaliacao]

In [38]:
# checando limpeza
print(f"Linhas: {df_avaliacoes_clean.shape[0]}")
print(f"Pedidos únicos: {df_avaliacoes_clean['order_id'].nunique()}")

Linhas: 95045
Pedidos únicos: 95045


In [39]:
# apagando colunas que não serão utilizadas
df_avaliacoes_clean = df_avaliacoes_clean.drop(columns=['review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp'])

In [40]:
# checando limpeza
print(f'Dados Brutos: {df_avaliacoes.shape}')
print(f'Dados Limpos: {df_avaliacoes_clean.shape}')

Dados Brutos: (99224, 7)
Dados Limpos: (95045, 3)


In [41]:
# salvando em data/processed
df_avaliacoes_clean.to_csv('../data/processed/avaliacoes_clean.csv', index=False)

### Registro da Limpeza - Tabela de Avaliações
- Colunas removidas: *review_comment_title*, *review_comment_message*, *review_creation_date* e *review_answer_timestamp* (7 -> 3 colunas)
- Filtro de acordo com a tabela de pedidos: 99.224 -> 96.095 linhas
- 95.568 pedidos com avaliação; 643 pedidos entregues sem avaliação (mantidos em pedidos e itens, porém naturalmente fora da satisfação)
- 523 pedidos com mais de uma avaliação removidos (1.050 linhas): 96.095 -> 95.045
- Resultado: 1 avaliação por pedido (95.045 linhas = 95.045 pedidos únicos)

## 5.5 Tabela de Produtos

In [42]:
# criando cópia dos dados raw para aplicar a limpeza
df_produtos_clean = df_produtos.copy()

In [43]:
# apagando colunas que não serão utilizadas
df_produtos_clean = df_produtos_clean.drop(columns=['product_name_lenght', 'product_description_lenght', 'product_photos_qty', 
                                                    'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm'])

In [44]:
# checando limpeza
print(f'Dados Brutos: {df_produtos.shape}')
print(f'Dados Limpos: {df_produtos_clean.shape}')

Dados Brutos: (32951, 9)
Dados Limpos: (32951, 2)


In [45]:
# checando nulos
df_produtos_clean['product_category_name'].isna().sum()

np.int64(610)

In [46]:
# preenchendo os produtos sem categoria com 'sem_categoria'
df_produtos_clean['product_category_name'] = df_produtos_clean['product_category_name'].fillna('sem_categoria')

In [47]:
# checando nulos (antes eram 610)
df_produtos_clean['product_category_name'].isna().sum()

np.int64(0)

In [48]:
# checando preenchimento 'sem_categoria'
(df_produtos_clean['product_category_name'] == 'sem_categoria').sum()

np.int64(610)

In [49]:
# salvando em data/processed
df_produtos_clean.to_csv('../data/processed/produtos_clean.csv', index=False)

### Registro da Limpeza - Tabela de Produtos
- Colunas removidas: *product_name_lenght*, *product_description_lenght*, *product_photos_qty*, *product_weight_g*, *product_length_cm*, *product_height_cm* e *product_width_cm* (9 -> 2 colunas)
- Os 610 produtos sem categoria que estavam com dados nulos foram preenchidos com 'sem_categoria'
    - Isso se faz necessário para que estes produtos fiquem visíveis ao fazer **groupby** nas análises
- O catálogo foi mantido (32.951 produtos), sem filtro de linhas

# 6.0 Arquivos Gerados

- Tabela de Avaliações: *avaliacoes_clean.csv* (95.045 x 3)
- Tabela de Clientes: *clientes_clean.csv* (96.211 x 3)
- Tabela de Itens dos Pedidos: *itens_clean.csv* (109.880 x 4)
- Tabela de Pedidos: *pedidos_clean.csv* (96.211 x 6)
- Tabela de Produtos: *produtos_clean.csv* (32.951 x 2)

Observação: ao carregar *pedidos_clean.csv*, usar **parse_dates** nas colunas *order_purchase_timestamp*, *order_delivered_customer_date* e *order_estimated_delivery_date*